# 04 · Clasificación multiclase con un dataset real (Wine)

## Objetivo
Aplicar todo lo aprendido en los notebooks anteriores a un **dataset real** con más de 2 clases, usando Scikit-learn para los datos y TensorFlow/Keras para el modelo — el mismo patrón que vimos en el tema "Scikit-learn vs TensorFlow": cada biblioteca hace lo que mejor sabe hacer.

Usaremos el dataset **Wine** (vinos) de scikit-learn, un dataset clásico similar en espíritu al de Iris, pero con distintos datos: características químicas de vinos, clasificados en 3 tipos de cultivo. (acutlaizado)

## Teoría: el flujo completo de un proyecto de clasificación multiclase

1. Cargar el dataset.
2. Separar en datos de entrenamiento y de prueba (`train_test_split`), para poder evaluar el modelo con datos que nunca vio.
3. Definir la arquitectura: la capa de entrada debe tener tantas neuronas como **características** tenga el dataset, y la capa de salida tantas neuronas como **clases** posibles, con activación `softmax`.
4. Compilar, entrenar, y evaluar con `argmax` (la clase con mayor probabilidad).


## Paso 1: Importar librerías

Aquí se ve claramente la combinación Scikit-learn + TensorFlow:
- `sklearn.datasets.load_wine`: para obtener el dataset real.
- `sklearn.model_selection.train_test_split`: para dividir los datos.
- `tensorflow` / `keras`: para construir y entrenar la red neuronal.


In [1]:
import tensorflow as tf
import numpy as np
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split

## Paso 2: Cargar y explorar el dataset

- `X = wine.data`: las características (2D — una fila por vino, una columna por característica química).
- `Y = wine.target`: las etiquetas (1D — el tipo de vino, como número entero: 0, 1 o 2).

Antes de construir el modelo, siempre conviene mirar cuántas características y cuántas clases hay — esos números definen el tamaño de la capa de entrada y de salida.


In [2]:
wine = load_wine()
X = wine.data
Y = wine.target

print("Forma de X (filas, columnas):", X.shape)
print("Cantidad de características (para la capa de entrada):", X.shape[1])
print("Clases posibles:", wine.target_names)
print("Cantidad de clases (para la capa de salida):", len(wine.target_names))

Forma de X (filas, columnas): (178, 13)
Cantidad de características (para la capa de entrada): 13
Clases posibles: ['class_0' 'class_1' 'class_2']
Cantidad de clases (para la capa de salida): 3


## Paso 3: Separar en entrenamiento y prueba

`train_test_split` reparte los datos en dos grupos:
- **80% entrenamiento**: con lo que el modelo aprende.
- **20% prueba**: datos que el modelo NUNCA ve durante el entrenamiento, para evaluar si realmente aprendió o solo "memorizó".

`random_state=42` fija la semilla aleatoria, para que el split sea siempre el mismo si volvemos a correr el notebook (reproducibilidad).


In [3]:
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42
)

print("Datos de entrenamiento:", X_train.shape[0])
print("Datos de prueba:", X_test.shape[0])

Datos de entrenamiento: 142
Datos de prueba: 36


## Paso 4: Construir la arquitectura

- Capa de entrada implícita: `input_shape=(13,)` porque el dataset Wine tiene 13 características químicas por vino.
- Dos capas ocultas con ReLU (12 y 8 neuronas), para darle más capacidad de aprendizaje que en los notebooks anteriores.
- Capa de salida: 3 neuronas (una por tipo de vino) con activación **softmax**, que convierte las salidas en una distribución de probabilidad que suma 1 entre las 3 clases (a diferencia de sigmoid, que da una sola probabilidad para 2 clases).


In [4]:
modelo = tf.keras.models.Sequential([
    tf.keras.layers.Dense(12, activation="relu", input_shape=(13,)),
    tf.keras.layers.Dense(8, activation="relu"),
    tf.keras.layers.Dense(3, activation="softmax")
])

modelo.summary()

c:\Users\ANT DOR\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 12)             │           168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │           104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 3)              │            27 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 299 (1.17 KB)

 Trainable params: 299 (1.17 KB)

 Non-trainable params: 0 (0.00 B)

## Paso 5: Compilar

- **loss="sparse_categorical_crossentropy"**: se usa cuando las etiquetas son números enteros (0, 1, 2), como en nuestro caso. Si las etiquetas estuvieran en formato one-hot (ej. [1,0,0]), usaríamos `categorical_crossentropy` en su lugar.


In [5]:
modelo.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

## Paso 6: Entrenar

Entrenamos solo con los datos de entrenamiento (`X_train`, `Y_train`), dejando el conjunto de prueba completamente aparte.


In [6]:
historial = modelo.fit(X_train, Y_train, epochs=100, verbose=0)
print("Pérdida final:", historial.history["loss"][-1])
print("Exactitud (accuracy) final:", historial.history["accuracy"][-1])

Pérdida final: 0.5289373397827148
Exactitud (accuracy) final: 0.7253521084785461


## Paso 7: Predecir una muestra individual

`softmax` entrega una probabilidad por cada una de las 3 clases. `argmax` nos dice cuál de esas 3 probabilidades es la más alta — esa es la clase que el modelo elige.


In [7]:
muestra = X_test[0].reshape(1, 13)
etiqueta_real = Y_test[0]

prediccion = modelo.predict(muestra)
print("Probabilidades por clase:", prediccion)
print("Clase predicha:", prediccion.argmax(1))
print("Clase real:", etiqueta_real)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step
Probabilidades por clase: [[0.89592445 0.00646485 0.09761066]]
Clase predicha: [0]
Clase real: 0


## Paso 8: Evaluar sobre todo el conjunto de prueba

Comparamos, de un vistazo, todas las predicciones contra todas las etiquetas reales del conjunto de prueba (el 20% que el modelo nunca vio entrenando).


In [8]:
predicciones = modelo.predict(X_test).argmax(1)

print("Predicciones:", predicciones)
print("Etiquetas reales:", Y_test)
print("Aciertos:", (predicciones == Y_test).sum(), "de", len(Y_test))

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
Predicciones: [0 0 2 0 1 0 1 2 1 0 0 2 0 1 0 1 1 1 0 1 0 0 2 2 2 2 1 2 1 0 0 1 0 0 0 0]
Etiquetas reales: [0 0 2 0 1 0 1 2 1 2 0 2 0 1 0 1 1 1 0 1 0 1 1 2 2 2 1 1 1 0 0 1 2 0 0 0]
Aciertos: 31 de 36


## Conclusión del notebook

- Vimos el ciclo completo de un proyecto de clasificación multiclase con un dataset real: cargar → dividir → construir → compilar → entrenar → evaluar.
- Scikit-learn se encargó de los datos (dataset + split); TensorFlow/Keras se encargó del modelo (red neuronal).
- La arquitectura de salida (`softmax` + `sparse_categorical_crossentropy`) es la misma "plantilla multiclase" que vimos en el notebook 03, aplicada ahora a un caso real.
